In [0]:
storage_account_name = "adlsgen1111"
storage_account_key = "PphoiAnr9BY7PAveVPLg/m+msI8DzTx4RJA6PDZJVg+iZ0ZivJR/4YLuS+/pKqOsKxOS/4WMx5k2+AStr72IYA=="
container_name = "silver"

In [0]:
mount_point = f"/mnt/{container_name}"
source_str = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/"
config_key = f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net"

try:
   
    if not any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        dbutils.fs.mount(
            source = source_str,
            mount_point = mount_point,
            extra_configs = {config_key: storage_account_key}
        )
        print(" Mounted successfully")
    else:
        print(" Already Mounted ")

except Exception as e:
    print(f"Error: {e}")

 Already Mounted 


In [0]:
df_bronze = spark.read.format("csv").option(
    "header", "true"
).option(
    "inferSchema", "true"
).load(
  "/mnt/bronze/smart city/*.csv"
)
print("Data Preview (Bronze):")
display(df_bronze)

Data Preview (Bronze):


region_id,timestamp,energy_consumption,building_type,population_density,historical_peak,ingestion_timestamp,source_file_name
R1,2025-12-01 00:00:00,249.96971495030178,Residential,6781,300,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R2,2025-12-01 00:00:00,240.96042507190316,Commercial,7086,450,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R3,2025-12-01 00:00:00,140.23544498565482,Residential,7624,400,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R4,2025-12-01 00:00:00,272.95353113516546,Residential,4746,440,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R1,2025-12-01 01:00:00,377.3899880441285,Residential,6781,300,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R2,2025-12-01 01:00:00,275.68765923228733,Commercial,7086,450,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R3,2025-12-01 01:00:00,213.5959006281118,Residential,7624,400,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R4,2025-12-01 01:00:00,125.03925490619515,Residential,4746,440,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R1,2025-12-01 02:00:00,271.90590754743124,Residential,6781,300,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv
R2,2025-12-01 02:00:00,364.94251544658346,Commercial,7086,450,2025-11-29T21:17:45.4677639Z,energy_dec_2025.csv


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType
from pyspark.sql.functions import col, to_timestamp, current_timestamp, month, dayofweek, hour, when, trim, lower, lit

# ---------------------------------------------------------
# 3. Schema Validation
# ---------------------------------------------------------
custom_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("region_id", StringType(), True),
    StructField("energy_consumption", StringType(), True), 

# ---------------------------------------------------------
# Transformation Logic
# ---------------------------------------------------------
df_silver = (
    df_bronze
    # 4.1 Remove Duplicates
    .dropDuplicates()
    
    # 4.4 Missing Value Handling 
    .dropna(subset=["timestamp", "energy_consumption"]) 
    
    # 4.2 Fix Timestamps
    .withColumn("reading_time", to_timestamp(col("timestamp")))
    
    # 4.3 String Standardization 
    .withColumn("region_id", lower(trim(col("region_id")))) 
    .withColumn("energy_consumption", col("energy_consumption").cast("double"))
    
    # أعمدة مساعدة للوقت
    .withColumn("month", month(col("reading_time")))
    .withColumn("weekday", dayofweek(col("reading_time"))) 
    .withColumn("hour", hour(col("reading_time")))
    
    # -----------------------------------------------------
    # 5. Feature Engineering
    # -----------------------------------------------------
    
    # 5.1 Peak Hours Flag 
    .withColumn("peak_hours_flag", 
                when((col("hour") >= 17) & (col("hour") <= 22), 1).otherwise(0))
    
    # 5.2 Weekend Flag
    .withColumn("weekend_flag",
                when(col("weekday").isin([6, 7]), 1).otherwise(0))
    
    
    .withColumn("processing_time", current_timestamp())
    .select(
        "reading_time",
        "region_id", 
        "energy_consumption", 
        "month", 
        "weekday",
        "hour",
        "peak_hours_flag", 
        "weekend_flag",     # New
        "processing_time"
    )
)

print("Successfully transformed with all features.")
display(df_silver)

Successfully transformed with all features.


reading_time,region_id,energy_consumption,month,weekday,hour,peak_hours_flag,weekend_flag,processing_time
2025-12-01T19:00:00Z,r4,191.87507868761503,12,2,19,1,0,2025-11-30T07:47:27.667087Z
2025-12-03T11:00:00Z,r3,122.52461227547553,12,4,11,0,0,2025-11-30T07:47:27.667087Z
2025-12-06T11:00:00Z,r2,209.3725227392904,12,7,11,0,1,2025-11-30T07:47:27.667087Z
2025-12-09T23:00:00Z,r2,319.57560474931597,12,3,23,0,0,2025-11-30T07:47:27.667087Z
2025-12-15T21:00:00Z,r2,125.50607151560212,12,2,21,1,0,2025-11-30T07:47:27.667087Z
2025-12-18T20:00:00Z,r4,269.1422654401336,12,5,20,1,0,2025-11-30T07:47:27.667087Z
2025-12-21T12:00:00Z,r4,227.34229576929008,12,1,12,0,0,2025-11-30T07:47:27.667087Z
2025-12-23T02:00:00Z,r1,198.24041582038038,12,3,2,0,0,2025-11-30T07:47:27.667087Z
2025-12-23T18:00:00Z,r3,320.98571293699354,12,3,18,1,0,2025-11-30T07:47:27.667087Z
2025-12-24T05:00:00Z,r2,155.08388709438609,12,4,5,0,0,2025-11-30T07:47:27.667087Z


In [0]:
ssave_path = "/mnt/silver/smart city/energy"
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(save_path)
print(f"saved to {save_path}")

saved to /mnt/silver/smart city/energy
